# Deploy MCP Servers on OpenShift

This notebook deploys MCP servers as shared services on OpenShift, accessible by all team members via Routes.

**Servers to deploy:**
1. Sequential Thinking — Structured problem solving (**fully local, always deployed**)
2. Context7 — Library documentation (requires internet, optional)
3. GitHub — Repository operations (requires internet, optional)
5. Playwright — Browser automation (requires internet, optional)
6. Code Sandbox — Secure code execution (**fully local, always deployed**)

> Internet-dependent servers auto-skip when connectivity is unavailable.

## 1. Verify Cluster Access

In [14]:
%%bash
echo "Cluster: $(oc whoami --show-server)"
echo "User: $(oc whoami)"
echo ""
echo "Apps domain (for Route URLs):"
oc get ingresses.config cluster -o jsonpath='{.spec.domain}'
echo ""

Cluster: https://api.openshift-cluster.sandbox1785.opentlc.com:6443
User: kube:admin

Apps domain (for Route URLs):
apps.openshift-cluster.sandbox1785.opentlc.com


## 2. Create Namespace and Secrets

All MCP servers deploy into the `mcp-servers` namespace.

In [15]:
%%bash
# Create namespace
oc apply -f manifests/00-namespace-secret.yaml

echo ""
echo "⚠️  IMPORTANT: Update the secret with your actual tokens:"
echo ""
echo "  oc set data secret/mcp-api-keys -n mcp-servers \\"
echo "    --from-literal=GITHUB_TOKEN=ghp_your-actual-token"

namespace/mcp-servers unchanged

⚠️  IMPORTANT: Update the secret with your actual tokens:

  oc set data secret/mcp-api-keys -n mcp-servers \
    --from-literal=GITHUB_TOKEN=ghp_your-actual-token


## 3. Server 1 — Context7 (Requires Internet)

Context7 provides up-to-date library documentation for AI agents.

The MCP server process can run locally, but it **always calls Upstash's external API** to fetch docs (the crawling/parsing backend is proprietary). This means:

| Environment | Works? | How |
|-------------|--------|-----|
| Internet access ✅ | Yes | Deploy as Pod in cluster or use remote endpoint directly |
| Air-gapped / disconnected ❌ | No | Not supported (Enterprise On-Premise license required) |

Below deploys Context7 as a Pod in the cluster (still needs outbound internet). **Skip this cell if disconnected.**

**Tools:** `resolve-library-id`, `get-library-docs`

In [16]:
%%bash
# Check internet connectivity, then deploy Context7 as a local Pod
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://mcp.context7.com/mcp)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "✅ Internet reachable — deploying Context7 as cluster Pod..."
    oc apply -f manifests/05-context7.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-context7 -n mcp-servers --timeout=120s 2>/dev/null \
        && echo "✅ Context7 Pod ready" \
        || echo "⏳ Pod still starting... check: oc get pods -n mcp-servers"
    echo ""
    echo "Route URL:"
    oc get route mcp-context7 -n mcp-servers -o jsonpath='https://{.spec.host}/mcp' 2>/dev/null
    echo ""
else
    echo "⚠️  No internet access (HTTP $HTTP_CODE) — skipping Context7."
    echo "   This server requires outbound connectivity to Upstash API."
    echo "   The rest of the lab works without it."
fi

Checking outbound internet access...
✅ Internet reachable — deploying Context7 as cluster Pod...


error: the path "manifests/05-context7.yaml" does not exist



deployment.apps/mcp-context7 condition met
✅ Context7 Pod ready

Route URL:
https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp


## 4. Server 2 — Playwright (Requires Internet)

Browser automation with Microsoft Playwright. Provides `navigate`, `click`, `fill`, `screenshot`, `pdf` tools for AI agents.

| Environment | Works? |
|-------------|--------|
| Internet access ✅ | Yes — can browse any website |
| Air-gapped / disconnected ❌ | Partial — only local/cluster URLs |

**Skip this cell if disconnected and no local web targets are needed.**

In [17]:
%%bash
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://www.google.com)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "301" ] || [ "$HTTP_CODE" = "302" ]; then
    echo "✅ Internet reachable — deploying Playwright MCP server..."
    # Remove old Chrome DevTools if present
    oc delete deployment mcp-chrome-devtools -n mcp-servers 2>/dev/null && echo "   (removed old chrome-devtools)"
    oc delete svc mcp-chrome-devtools -n mcp-servers 2>/dev/null
    oc delete route mcp-chrome-devtools -n mcp-servers 2>/dev/null

    oc apply -f manifests/04-playwright.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-playwright -n mcp-servers --timeout=180s 2>/dev/null \
        && echo "✅ Playwright MCP ready" \
        || echo "⏳ Pod still starting (Playwright image is large)..."
    echo ""
    echo "Route URL:"
    oc get route mcp-playwright -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "⚠️  No internet access (HTTP $HTTP_CODE) — skipping Playwright."
    echo "   This server needs internet to browse external websites."
    echo "   The rest of the lab works without it."
fi

Checking outbound internet access...
✅ Internet reachable — deploying Playwright MCP server...


error: the path "manifests/04-playwright.yaml" does not exist



deployment.apps/mcp-playwright condition met
✅ Playwright MCP ready

Route URL:
https://mcp-playwright-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp


## 5. Server 3: Code Sandbox (Local — Always Deployed)

Secure code execution sandbox powered by OpenShell (NVIDIA + Red Hat).
Runs Python, Bash, and Node.js code in an isolated workspace.
**No internet required** — works fully offline.

In [18]:
%%bash
echo "Deploying Code Sandbox MCP server (local, no internet needed)..."
oc apply -f manifests/06-code-sandbox.yaml

echo ""
oc wait --for=condition=available deployment/mcp-code-sandbox -n mcp-servers --timeout=180s 2>/dev/null \
    && echo "✅ Code Sandbox MCP ready" \
    || echo "⏳ Pod still starting..."

echo ""
echo "Route URL:"
oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""
echo ""
echo "Health check:"
ROUTE=$(oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='{.spec.host}')
curl -sk "https://${ROUTE}/health"
echo ""

Deploying Code Sandbox MCP server (local, no internet needed)...


error: the path "manifests/06-code-sandbox.yaml" does not exist



deployment.apps/mcp-code-sandbox condition met
✅ Code Sandbox MCP ready

Route URL:
https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp

Health check:
{"status":"ok","server":"code-sandbox"}


## 6. Server 4 — DuckDuckGo Search (Requires Internet)

Provides web search and content fetching via [DuckDuckGo](https://github.com/nickclyde/duckduckgo-mcp-server). No API key required.

**Tools:**
- `search` — Web search with LLM-friendly formatted results
- `fetch_content` — Fetch and parse webpage content

In [ ]:
%%bash
echo "Deploying DuckDuckGo MCP server (requires internet for PyPI + DDG)..."
oc apply -f manifests/04-duckduckgo.yaml

echo ""
oc wait --for=condition=available deployment/mcp-duckduckgo -n mcp-servers --timeout=180s 2>/dev/null \
    && echo "✅ DuckDuckGo MCP ready" \
    || echo "⏳ Pod still starting (Python venv install takes ~60s)..."

echo ""
echo "Route URL:"
oc get route mcp-duckduckgo -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""
echo ""
echo "Health check:"
ROUTE=$(oc get route mcp-duckduckgo -n mcp-servers -o jsonpath='{.spec.host}')
curl -sk "https://${ROUTE}/health"
echo ""

## 7. Verify All Servers

In [19]:
%%bash
echo "MCP Server Deployment Status"
echo "============================================================"
echo ""
echo "=== Pods ==="
oc get pods -n mcp-servers -o wide

echo ""
echo "=== Routes (IDE Endpoints) ==="
echo ""
printf "%-25s %s\n" "SERVER" "ENDPOINT"
printf "%-25s %s\n" "-------" "--------"

for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    printf "%-25s %s\n" "$route" "https://${host}/mcp"
done

MCP Server Deployment Status

=== Pods ===
NAME                                READY   STATUS    RESTARTS      AGE   IP            NODE                                         NOMINATED NODE   READINESS GATES
mcp-code-sandbox-78d984cb56-25b9b   1/1     Running   0             27h   10.131.2.86   ip-10-0-100-123.us-east-2.compute.internal   <none>           <none>
mcp-context7-58d87b698-l85zx        1/1     Running   2 (27h ago)   27h   10.129.2.33   ip-10-0-143-166.us-east-2.compute.internal   <none>           <none>
mcp-playwright-99d967458-pssww      1/1     Running   0             27h   10.128.2.69   ip-10-0-162-87.us-east-2.compute.internal    <none>           <none>

=== Routes (IDE Endpoints) ===

SERVER                    ENDPOINT
-------                   --------
mcp-code-sandbox          https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
mcp-context7              https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.c

In [20]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

print("Health Check (deployed servers):")
print("=" * 60)

# Streamable HTTP MCP servers only accept POST on /mcp
init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
})

for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/mcp"
        r = subprocess.run(
            ["curl", "-sk", "-X", "POST",
             "-H", "Content-Type: application/json",
             "-H", "Accept: application/json, text/event-stream",
             "-d", init_payload,
             "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
            capture_output=True, text=True)
        status = "✅" if r.stdout.strip() == "200" else "❌"
        print(f"{status} {name}: {url}")

if not result.stdout.strip():
    print("⚠️  No routes found. Deploy servers first.")

Health Check (deployed servers):
✅ mcp-code-sandbox: https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
✅ mcp-context7: https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
✅ mcp-playwright: https://mcp-playwright-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp


## Summary

| Server | Tools | Internet Required | Route |
|--------|-------|:-----------------:|-------|
| Context7 | `resolve-library-id`, `get-library-docs` | ✅ Yes | `https://mcp-context7-mcp-servers.apps.CLUSTER/mcp` |
| Playwright | `browser_navigate`, `browser_screenshot`, ... | ✅ Yes | `https://mcp-playwright-mcp-servers.apps.CLUSTER/mcp` |
| Code Sandbox | `execute_code`, `read_file`, `write_file`, `list_files` | ❌ No | `https://mcp-code-sandbox-mcp-servers.apps.CLUSTER/mcp` |
| DuckDuckGo | `search`, `fetch_content` | ✅ Yes | `https://mcp-duckduckgo-mcp-servers.apps.CLUSTER/mcp` |

All servers use **Streamable HTTP** transport on `/mcp` endpoint.

> **Disconnected 환경에서는**: Code Sandbox만 단독 배포 가능.

## Next Steps

→ `3_connect_ide_clients.ipynb` — Configure your IDE to use these MCP server Routes (direct access)
→ `../2_ai_gateway/2_enable_maas.ipynb` — Register MCP servers with MaaS gateway for unified access with auth